# Day 48 · Tokens: what the model reads, and why Hinglish costs more

**First-principles question:** Why does the same message cost more in some languages?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import pandas as pd
import requests
import tiktoken

ENCODINGS = {"o200k_base": tiktoken.get_encoding("o200k_base"), "cl100k_base": tiktoken.get_encoding("cl100k_base")}
OLLAMA, MODEL = "http://localhost:11434", "qwen3:4b"

## 1 · One message, many scripts
Write the same support message in each language (your own words).

In [ ]:
MESSAGES = {
    "English": "My internet has been down since morning. Please send a technician today.",
    "Hinglish": "Mera internet subah se band hai. Please aaj technician bhej do.",
    # TODO: the same message in Hindi (Devanagari), Gujarati, and at least 4 more rows:
    #       a long English email, a Hinglish one with emoji, a Marathi or Tamil one, an email full of numbers…
}

## 2 · Count the tokens three ways
Two OpenAI tokenizers (tiktoken) and your own Ollama model. `raw: true` sends the text without a chat template, so `prompt_eval_count` is just your text (plus a start token). Ollama can reuse tokens it has cached from the previous prompt: if a number looks too small, run the cell again.

In [ ]:
def ollama_tokens(text):
    body = {"model": MODEL, "prompt": text, "raw": True, "stream": False, "options": {"num_predict": 1}}
    return requests.post(f"{OLLAMA}/api/generate", json=body, timeout=120).json()["prompt_eval_count"]

rows = []
for language, text in MESSAGES.items():
    row = {"language": language, "characters": len(text)}
    # TODO: for each encoding in ENCODINGS, row[name] = len(encoding.encode(text))
    row["ollama"] = ollama_tokens(text)
    rows.append(row)
table = pd.DataFrame(rows).set_index("language")

## 3 · Characters per token, and cost compared to English

In [ ]:
table["chars per token (o200k)"] = (table["characters"] / table["o200k_base"]).round(2)
table["cost vs English (o200k)"] = (table["o200k_base"] / table.loc["English", "o200k_base"]).round(2)
table

**My budget recommendation for an Indian customer:** …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert len(MESSAGES) >= 8, f"Step 1: at least 8 messages ({len(MESSAGES)} so far)"
assert {"o200k_base", "cl100k_base", "ollama"} <= set(table.columns), "Step 2: three token counts per row"
assert (table[["o200k_base", "cl100k_base"]] > 0).all().all()
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")